# Model Construction Notebook (Decision Tree)
## CS 171 Final Project: West Coast Swing Dance Pattern Classification

**Author:** Nguyen Pham

This notebook covers the training and evaluation of the Decision Tree model:
- Load preprocessed keypoint data
- Engineer features (angles, velocities, aggregated statistics)
- Train Decision Tree classifier
- Evaluate model performance
- Save trained model pipeline

In [51]:
from pathlib import Path
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

In [52]:
# paths and constants
DATA_DIR = Path('../data/keypoints_1')  
OUT_DIR = Path('../data/processed')
LABELS_CSV = Path('../data/hp.csv')  
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Load Data

In [53]:
def load_labels_mapping():
    if not LABELS_CSV.exists():
        print(f"Warning: {LABELS_CSV} not found")
        return None
    
    labels_df = pd.read_csv(LABELS_CSV)
    mapping = {}
    for _, row in labels_df.iterrows():
        video_name = f"{row['division']}_{row['id']}"
        mapping[video_name] = row['labels']
    return mapping

In [54]:
def find_videos(root=DATA_DIR, labels_csv=LABELS_CSV):
    records = []
    
    if not labels_csv.exists():
        raise FileNotFoundError(f"Labels CSV not found: {labels_csv}")
    if not root.exists():
        raise FileNotFoundError(f"Data dir not found: {root}")
    df = pd.read_csv(labels_csv)
    print(f"hp.csv contains {len(df)} labeled videos")
    
    missing_videos = []
    
    for _, row in df.iterrows():
        division = row['division']
        video_id = row['id']
        video_name = f"{division}_{video_id}"
        trick_label = row['labels']
        
        # Build expected path
        np_path = root / division / video_name / 'keypoints.npy'
        json_path = root / division / video_name / 'keypoints.json'
        
        if np_path.exists():
            records.append({
                'label': trick_label,
                'division': division,
                'video': video_name,
                'np_path': np_path,
                'json_path': json_path
            })
        else:
            missing_videos.append(video_name)
    
    if missing_videos:
        print(f"Warning: {len(missing_videos)} videos from hp.csv not found:")
        for v in missing_videos[:5]:
            print(f"  - {v}")
        if len(missing_videos) > 5:
            print(f"  ... and {len(missing_videos) - 5} more")
    
    print(f"Loaded {len(records)} videos (matching hp.csv)")
    return pd.DataFrame(records)

In [55]:
def load_keypoints(np_path: Path):
    arr = np.load(np_path)
    if arr.ndim != 3: 
        raise ValueError(f"Unexpected shape {np_path}: {arr.shape}")
    return arr.astype(np.float32)

## Preprocessing Functions

In [56]:
def interpolate_missing(arr, visibility_threshold=0.1):
    # fill in missing keypoints using interpolation
    out = arr.copy()
    T = out.shape[0]
    
    for lm in range(out.shape[1]):
        vis = out[:, lm, 3]
        bad = vis < visibility_threshold
        if bad.all():
            continue
        
        for d in range(3):
            vals = out[:, lm, d].astype(float)
            vals[bad] = np.nan
            s = pd.Series(vals)
            vals_filled = s.interpolate(limit_direction='both').bfill().ffill().values
            out[:, lm, d] = vals_filled.astype(np.float32)
    
    return out

In [57]:
def torso_scale_single(lm):
    # get torso length for normalization
    left_sh = lm[11][:2]
    right_sh = lm[12][:2]
    left_hip = lm[23][:2]
    right_hip = lm[24][:2]
    mid_sh = (left_sh + right_sh) / 2.0
    mid_hip = (left_hip + right_hip) / 2.0
    d = np.linalg.norm(mid_sh - mid_hip)
    return max(d, 1e-6)

In [58]:
def normalize_by_torso(arr):
    # normalize positions relative to hip center and torso length
    out = arr.copy()
    for t in range(out.shape[0]):
        lm = out[t]
        hip_center = (lm[23][:2] + lm[24][:2]) / 2.0
        scale = torso_scale_single(lm)
        out[t, :, :2] = (lm[:, :2] - hip_center) / scale
    return out

In [59]:
def pad_or_truncate(arr, target_len=64, strategy='center'):
    T = arr.shape[0]
    if T == target_len:
        return arr
    if T > target_len:
        if strategy == 'center':
            start = max(0, (T - target_len) // 2)
            return arr[start:start+target_len]
        return arr[:target_len]
    
    pad = np.zeros((target_len - T, arr.shape[1], arr.shape[2]), dtype=arr.dtype)
    return np.concatenate([arr, pad], axis=0)

In [60]:
def angle_at(a, b, c):
    # compute angle at point b formed by points a-b-c
    ba = a - b
    bc = c - b
    lena = np.linalg.norm(ba)
    lenb = np.linalg.norm(bc)
    if lena < 1e-6 or lenb < 1e-6:
        return 0.0
    cosang = np.dot(ba, bc) / (lena * lenb)
    cosang = np.clip(cosang, -1.0, 1.0)
    return math.acos(cosang)

## Feature Extraction

In [61]:
ANGLE_TRIPLETS = [(11, 13, 15), (12, 14, 16), (23, 25, 27), (24, 26, 28)]
def compute_frame_features(seq):
    # extract features from keypoint sequence
    T = seq.shape[0]
    flat = seq[:, :, :3].reshape(T, -1)
    
    # joint angles
    angles = np.zeros((T, len(ANGLE_TRIPLETS)), dtype=np.float32)
    for t in range(T):
        for i, (a, b, c) in enumerate(ANGLE_TRIPLETS):
            angles[t, i] = angle_at(seq[t, a, :2], seq[t, b, :2], seq[t, c, :2])
    
    # pairwise distances between key joints
    important = [11, 12, 23, 24, 13, 14, 25, 26]
    pdists = []
    for t in range(T):
        coords = seq[t, important, :2]
        dists = []
        for i in range(coords.shape[0]):
            for j in range(i+1, coords.shape[0]):
                dists.append(np.linalg.norm(coords[i] - coords[j]))
        pdists.append(dists)
    pdists = np.array(pdists)
    
    # velocity
    vel = np.vstack((np.zeros((1, flat.shape[1])), np.diff(flat, axis=0)))
    
    feats = np.concatenate([flat, angles, pdists, vel], axis=1)
    return feats

In [62]:
def aggregate_video_features(seq, target_len=64):
    # preprocess and aggregate features for entire video
    seq = interpolate_missing(seq)
    seq = normalize_by_torso(seq)
    seq = pad_or_truncate(seq, target_len=target_len)
    
    frame_feats = compute_frame_features(seq)
    mean = frame_feats.mean(axis=0)
    std = frame_feats.std(axis=0)
    
    return np.concatenate([mean, std])

In [63]:
def build_aggregated_dataset(index_df, sample_limit=None, target_len=64):
    rows = []
    X = []
    y = []
    
    subset = index_df.sample(min(len(index_df), sample_limit), random_state=0) if sample_limit is not None else index_df
    
    for _, r in subset.iterrows():
        seq = load_keypoints(r['np_path'])
        vec = aggregate_video_features(seq, target_len=target_len)
        X.append(vec)
        y.append(r['label'])
        rows.append({
            'label': r['label'],
            'division': r.get('division', ''),
            'video': r['video'],
            'np_path': str(r['np_path'])
        })
    
    return np.array(X), np.array(y), pd.DataFrame(rows)

## Run Analysis

In [64]:
# build dataset
df = find_videos()
X, y, meta = build_aggregated_dataset(df, target_len=64)

# Filter for target classes only
target_classes = ['sugar_push', 'sugar_tag']
mask = np.isin(y, target_classes)
X = X[mask]
y = y[mask]


hp.csv contains 20 labeled videos
  - advanced_11
Loaded 19 videos (matching hp.csv)


## Decision Tree Classification

In [65]:
# Split data into train and test sets
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, np.arange(len(y)), test_size=0.2, random_state=17, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# Train Decision Tree Classifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

# Evaluate
y_pred = dt.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"\nDecision Tree Accuracy: {acc:.2%}")


Training samples: 15
Testing samples: 4

Decision Tree Accuracy: 100.00%


## Output a Model

In [66]:
# Save model
model_path = OUT_DIR / 'decision_tree_classifier.joblib'
joblib.dump(dt, model_path)
print(f"Saved Decision Tree model to {model_path}")


Saved Decision Tree model to ../data/processed/decision_tree_classifier.joblib
